# Bronze - Actors Data

## Setup Environment and Prepare data 

In [1]:
from letterboxd_data_pipeline.load_data import load_files_from_layer
from letterboxd_data_pipeline.explore_data import is_possible_na
import numpy as np

bronze_df = load_files_from_layer(layer="bronze", file_list=["actors.parquet"])
bronze_actors_df = bronze_df["actors"]

Start loading data...
Loading: data/bronze/actors.parquet
Done!
Finish loading.


## Simple Explore

In [2]:
bronze_actors_df.describe(include="all")

,id,name,role
count,5798450,5798446,4436891
unique,634302,1600662,1920085
top,1123471,Mel Blanc,Self
freq,555,1058,188502


In [3]:
bronze_actors_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5798450 entries, 0 to 5798449
Data columns (total 3 columns):
 #   Column  Dtype 
---  ------  ----- 
 0   id      object
 1   name    object
 2   role    object
dtypes: object(3)
memory usage: 132.7+ MB


In [4]:
bronze_actors_df.head(10)

,id,name,role
0,1000001,Margot Robbie,Barbie
1,1000001,Ryan Gosling,Ken
2,1000001,America Ferrera,Gloria
3,1000001,Ariana Greenblatt,Sasha
4,1000001,Issa Rae,Barbie
5,1000001,Kate McKinnon,Barbie
6,1000001,Alexandra Shipp,Barbie
7,1000001,Emma Mackey,Barbie
8,1000001,Hari Nef,Barbie
9,1000001,Sharon Rooney,Barbie


## Number of NA

In [5]:
bronze_actors_df.isna().sum()

id            0
name          4
role    1361559
dtype: int64

In [6]:
bronze_actors_df[bronze_actors_df["name"].isna()]

,id,name,role
4145738,1443629,None,None
4281100,1469981,None,Self
4306960,1474958,None,Cinematography
5430275,1773264,None,None


## Possible NA

In [7]:
possible_na_df = bronze_actors_df[is_possible_na(bronze_actors_df["name"]) | is_possible_na(bronze_actors_df["role"])]
possible_na_df

,id,name,role
18917,1000301,Judy Kuhn,Nan
80182,1001571,Cordelia Richards,Nan
109214,1002291,Mavis Paenga,Nan
128884,1002801,Connie Danese,Nan
159169,1003601,Eileen Colgan,Nan
...,...,...,...
5461595,1783487,Zillah Bateman,Nan
5652329,1860568,Na,None
5656233,1862670,Christine Uhebe,Nan
5787871,1935653,Marie-Philomène Nga,Na


In [8]:
possible_na_df.describe(include="all")

,id,name,role
count,174,174,173
unique,172,168,3
top,1330379,Catherine Tate,Nan
freq,2,3,154


### Possible NA group by Role

In [9]:
possible_na_df.groupby(by="role").agg(count=("id", "count"))

,count
role,
A Letterboxd User,1
Na,18
Nan,154


### Possible NA name

In [10]:
possible_na_df[is_possible_na(possible_na_df["name"])]

,id,name,role
5652329,1860568,Na,None
5792614,1938925,Na,A Letterboxd User


## Most high risk schizophrenic actor (One actor in multiple character per movie)

In [11]:
talented_actor_df = bronze_actors_df.copy(True)

grouped_source_df = talented_actor_df.groupby(by=["id", "name"])

talented_actor_df["role_count"] = grouped_source_df["name"].transform(np.size)

talented_actor_df = talented_actor_df[
    ["id", "name", "role", "role_count"]
].sort_values(["id", "role_count", "name"], ascending=[True, False, True])

talented_actor_df

,id,name,role,role_count
147,1000001,Aaron J. Smith,Dancer,1.0
88,1000001,Adam Blaug,Dancer,1.0
100,1000001,Adam Crossley,Dancer,1.0
110,1000001,Adam Fogarty,Dancer,1.0
144,1000001,Adam Paul Robertson,Dancer,1.0
...,...,...,...,...
5798442,1941596,Nick Cheung,Zhang Yao/张耀,1.0
5798444,1941596,Sandrine Pinna,None,1.0
5798446,1941596,线雨轩,Tata/塔塔,1.0
5798448,1941597,Hiroshi Mikami,None,1.0
